In [19]:
import torch
import soundfile as sf
from qwen_asr import Qwen3ASRModel
import pandas as pd

#
AudioPATH= "./audio"
ModelId1="Qwen/Qwen3-ASR-1.7B"
ModelId2="Qwen/Qwen3-ASR-0.6B "

In [20]:
torch.cuda.is_available()

True

In [21]:
df = pd.read_parquet("train-00000-of-00001.parquet",  engine='pyarrow')

In [22]:
df.columns

Index(['audio', 'sentence', 'topic'], dtype='object')

In [23]:
df

,audio,sentence,topic
0,{'bytes': b'ID3\x04\x00\x00\x00\x00\x00#TSSE\x...,你知唔知今日嘅temperature會落到低至10度，一定要著厚d嘅clothes啊！,天氣
1,{'bytes': b'ID3\x04\x00\x00\x00\x00\x00#TSSE\x...,我check咗天氣app，佢話今日會有thunderstorm，最好唔好去戶外行嘅。,天氣
2,{'bytes': b'ID3\x04\x00\x00\x00\x00\x00#TSSE\x...,飛機已經被grounded喇，因為typhoon signal已經升到No.8嚟啦。,天氣
3,{'bytes': b'ID3\x04\x00\x00\x00\x00\x00#TSSE\x...,周圍都有well-developed霧，嚴重limit我哋嘅visibility，開車嘅時候...,天氣
4,{'bytes': b'ID3\x04\x00\x00\x00\x00\x00#TSSE\x...,Forecast話晚上嘅weather會變得好cold，會降到freezing point，...,天氣
...,...,...,...
14012,{'bytes': b'ID3\x04\x00\x00\x00\x00\x00#TSSE\x...,雖然香港天氣好hot，但我哋可以想下use less air con，改用風扇。,环境
14013,{'bytes': b'ID3\x04\x00\x00\x00\x00\x00#TSSE\x...,Buy local 嘅商品都可以減少carbon footprint，唔再郵寄長途嘅貨品。,环境
14014,{'bytes': b'ID3\x04\x00\x00\x00\x00\x00#TSSE\x...,用solar power同wind power都唔錯，我哋要追求sustainable嘅生活方式。,环境
14015,{'bytes': b'ID3\x04\x00\x00\x00\x00\x00#TSSE\x...,"Reduce, Reuse, Recycle 呢個三R原則喺日常生活中實踐，即係好好嘅典範。",环境


In [8]:

model = Qwen3ASRModel.from_pretrained(
    ModelId1,
    dtype=torch.bfloat16,
    device_map="cuda:0",
    # attn_implementation="flash_attention_2",
    attn_implementation= "sdpa",
    max_inference_batch_size=32, # Batch size limit for inference. -1 means unlimited. Smaller values can help avoid OOM.
    max_new_tokens=256, # Maximum number of tokens to generate. Set a larger value for long audio input.
)

Loading checkpoint shards: 100%|██████████████████| 2/2 [00:01<00:00,  1.07it/s]


In [9]:
model

In [11]:

results = model.transcribe(
    audio="https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen3-ASR-Repo/asr_en.wav",
    language=None, # set "English" to force the language
)

print(results[0].language)
print(results[0].text)

Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


English
Hmm. Oh yeah, yeah. He wasn't even that big when I started listening to him, but and his solo music didn't do overly well, but he did very well when he started writing for other people.


In [15]:
supportedLang = model.get_supported_languages()
supportedLang

['Chinese',
 'English',
 'Cantonese',
 'Arabic',
 'German',
 'French',
 'Spanish',
 'Portuguese',
 'Indonesian',
 'Italian',
 'Korean',
 'Russian',
 'Thai',
 'Vietnamese',
 'Japanese',
 'Turkish',
 'Hindi',
 'Malay',
 'Dutch',
 'Swedish',
 'Danish',
 'Finnish',
 'Polish',
 'Czech',
 'Filipino',
 'Persian',
 'Greek',
 'Romanian',
 'Hungarian',
 'Macedonian']

In [16]:
result = model.transcribe(
    audio=f"{AudioPATH}/1.wav",
    language=None,
)

Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


In [17]:
result.

[ASRTranscription(language='Cantonese', text='我check咗天氣App，佢話今日會有Thunderstorm，最好唔好去戶外行嘅。', time_stamps=None)]

In [24]:
df["sentence"][1]

'我check咗天氣app，佢話今日會有thunderstorm，最好唔好去戶外行嘅。'